# Task 1 — Traffic model: estimate real visitor count

Build a sensor-to-footfall model calibrated against manual counting windows (assignment Task 1).

This notebook compares three clearly separated approaches:

- **Plan A** — trusted-device linear baseline (`footfall_plan_a` / `footfall_trusted_linear`)
- **Plan B** — capture rate + HOD v2 (`footfall_plan_b`) + mall visitors (`estimated_mall_visitors`)
- **Plan C** — trend-preserving clean-device elasticity (`footfall_plan_c`, delivered estimate)

**Workflow:** compute all plans from scratch → show summary tables → interactive Plan A/B/C dashboards → static plots.

Plots: `outputs/plots/task1/plan_a/`, `outputs/plots/task1/plan_b/`

In [ ]:
from IPython.display import display

from notebook_helpers import (
    load_facilities,
    plot_task1,
    plot_task1_abc_interactive,
    prepare_task1_abc,
    run_task1,
    show_task_plots,
)

# Compute Task 1 from scratch (Plan A + Plan B; Plan C is produced in Plan A pipeline).
FORCE_RECOMPUTE = True

t1 = run_task1(force_recompute=FORCE_RECOMPUTE, plan="both")
plan_a = t1["plan_a"]
plan_b = t1["plan_b"]

# Ensure rich objects exist when loading from cache.
if "calibration" not in plan_a or "task1_result" not in plan_a:
    from notebook_helpers import run_plan_a
    plan_a = run_plan_a(force_recompute=True, write_outputs=True)
if "hourly" not in plan_b:
    from notebook_helpers import run_plan_b
    plan_b = run_plan_b(force_recompute=True, write_outputs=True)

task1_abc = prepare_task1_abc(plan_a, plan_b)
facilities = load_facilities()

display(task1_abc["overall_summary"])

mall_visitors = task1_abc.get("mall_visitors_weekly")
if mall_visitors == mall_visitors:
    display({"mall_visitors_weekly": mall_visitors})

display(task1_abc["facility_weekly"].head(10))

In [ ]:
display(task1_abc["daily_compare"])
display(task1_abc["calibration_compare"])

if not task1_abc["mall_daily"].empty:
    display(task1_abc["mall_daily"])

## Interactive dashboards — Plan A / Plan B / Plan C

- Daily mall footfall (all three plans) + mall visitors
- Weekly footfall by location
- Daily footfall by location (dropdown)
- Manual calibration windows
- Hourly mall visitors

In [ ]:
plot_task1_abc_interactive(task1_abc, facilities=facilities)

## Static plots (saved PNGs)

In [ ]:
plot_task1(plan="both")
show_task_plots(1, plan="a")
show_task_plots(1, plan="b")

## Recommendation

**Plan C** is the best option for assignment delivery: it preserves realistic hourly/daily trend shape while staying calibrated to manual counts.

Plan A has lower calibration error on only four manual windows, but the trusted-linear model is flatter over time because its intercept dominates low-volume hours. Plan B is useful for comparison and mall-level deduplicated visitors, but the selected Task 1 delivery estimate remains Plan C.